# contiguous-layout — worked example 3: Show why outer-axis slices stay contiguous but inner ones break it

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `contiguous-layout`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Slicing a *leading* (outer) axis with `step=1` keeps a tensor contiguous: you just trim whole blocks off the front, and the remaining elements are still one unbroken run. Slicing an *inner* axis to a partial extent leaves gaps between the surviving rows, so the view becomes non-contiguous even though no axis was permuted.

## Worked solution

Start from a contiguous base `t.arange(60).reshape(3, 4, 5)`, whose strides are `(20, 5, 1)`.

**Step 1 — slice the outer axis.** `base[1:3]` keeps axes 1 and 2 whole and drops the first `(4, 5)` block. Memory-wise this is just a contiguous sub-run starting at offset `20`; the strides `(20, 5, 1)` still satisfy the row-major formula for the new shape `(2, 4, 5)`, so `is_contiguous()` is `True`.

**Step 2 — slice an inner axis.** `base[:, :, 1:4]` keeps only 3 of the 5 elements along the last axis. The surviving elements of each row are adjacent, but between one row and the next there is now a 2-element gap (we skipped indices 0 and 4). The stride of the last axis is still `1`, yet the middle axis's stride `5` no longer equals `length_of_last_axis = 3`. The contiguous invariant `stride[k] == stride[k+1] * shape[k+1]` is violated, so `is_contiguous()` is `False`.

**Step 3 — the takeaway.** Contiguity survives trimming the *outermost* varying axis but breaks the moment an *inner* axis is shrunk, because inner shrinking introduces strides that no longer chain into a single packed buffer.

In [ ]:
base = t.arange(60).reshape(3, 4, 5)

outer_slice = base[1:3]            # trims outer axis, step 1
inner_slice = base[:, :, 1:4]      # shrinks inner axis -> gaps

print("base strides:", tuple(base.stride()))
print("outer slice shape/contig:", tuple(outer_slice.shape), outer_slice.is_contiguous())
print("inner slice shape/contig:", tuple(inner_slice.shape), inner_slice.is_contiguous())